In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col,sum,count

In [ ]:
spark = ( #Cria uma sessão Spark.
    SparkSession.builder
    .appName("AnaliseMatriculas")
    .master("local[*]")
    .getOrCreate()
)

dados = [ #Constrói um DataFrame com dados fictícios.
    ("Arquitetura e Urbanismo", 100, 5000), #Agrupa os registros por curso.
    ("Engenharia de Computação", 120, 6000),
    ("Engenharia Elétrica", 80, 4000),
    ("Engenharia Mecânica", 90, 5500),
    ("Engenharia Química", 70, 3500),
]

colunas = ["curso" , "matriculas" , "receita"]

df = spark.createDataFrame(dados,colunas)

df.show()

resultado = ( #Calcula o total de matrículas e a quantidade de registros de cada curso.
    df.groupby("curso")
    .agg(
        sum("matriculas").alias("total_matriculas"),
        count("*").alias("registros")
    )
)

resultado.show()

+--------------------+----------+-------+
|               curso|matriculas|receita|
+--------------------+----------+-------+
|Arquitetura e Urb...|       100|   5000|
|Engenharia de Com...|       120|   6000|
| Engenharia Elétrica|        80|   4000|
| Engenharia Mecânica|        90|   5500|
|  Engenharia Química|        70|   3500|
+--------------------+----------+-------+

+--------------------+----------------+---------+
|               curso|total_matriculas|registros|
+--------------------+----------------+---------+
|Arquitetura e Urb...|             100|        1|
|Engenharia de Com...|             120|        1|
| Engenharia Mecânica|              90|        1|
| Engenharia Elétrica|              80|        1|
|  Engenharia Química|              70|        1|
+--------------------+----------------+---------+



In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [ ]:

import pandas as pd

pdf = pd.DataFrame({
    "curso": ["Arquitetura e Urbanismo", "Engenharia de Computação", "Engenharia Elétrica", "Engenharia Mecânica", "Engenharia Química"],
    "matriculas": [100, 120, 80, 90, 70],
    "receita": [5000, 6000, 4000, 5500, 3500]
})

print(pdf.groupby("curso")["matriculas"].sum())

curso
Arquitetura e Urbanismo     100
Engenharia Elétrica          80
Engenharia Mecânica          90
Engenharia Química           70
Engenharia de Computação    120
Name: matriculas, dtype: int64


In [ ]:
#com Pandas-on-Spark
import pyspark.pandas as ps

# Desativa o erro referente ao modo ANSI ativo no Spark
ps.set_option("compute.fail_on_ansi_mode", False)

psdf = ps.DataFrame({
    "curso": ["Arquitetura e Urbanismo", "Engenharia de Computação", "Engenharia Elétrica", "Engenharia Mecânica", "Engenharia Química"],
    "matriculas": [100, 120, 80, 90, 70],
})

print(psdf.groupby("curso")["matriculas"].sum())

/usr/local/lib/python3.13/dist-packages/pyspark/pandas/utils.py:1037: PandasAPIOnSparkAdviceWarning: The config 'spark.sql.ansi.enabled' is set to True. This can cause unexpected behavior from pandas API on Spark since pandas API on Spark follows the behavior of pandas, not SQL.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


curso
Arquitetura e Urbanismo     100
Engenharia de Computação    120
Engenharia Mecânica          90
Engenharia Elétrica          80
Engenharia Química           70
Name: matriculas, dtype: int64


In [ ]:
# Criar uma visualização temporária a partir do DataFrame 'df' existente
df.createOrReplaceTempView("vendas")

# Executar a consulta SQL diretamente
resultado_sql = spark.sql("""
SELECT
    curso,
    SUM(receita) AS faturamento
FROM vendas
GROUP BY curso
ORDER BY faturamento DESC
""")

# Exibir o resultado
resultado_sql.show()

+--------------------+-----------+
|               curso|faturamento|
+--------------------+-----------+
|Engenharia de Com...|       6000|
| Engenharia Mecânica|       5500|
|Arquitetura e Urb...|       5000|
| Engenharia Elétrica|       4000|
|  Engenharia Química|       3500|
+--------------------+-----------+



In [ ]:
%pip install -q pyspark

In [ ]:
from pyspark.sql import SparkSession, functions as F



In [ ]:
spark = (
    SparkSession.builder
    .appName("SistemaMatriculas")
    .master("local[*]")
    .getOrCreate()
)

# 1. Dados de exemplo - Renomeando 'nome' para 'nome_aluno'
alunos = spark.createDataFrame([
    (1, "Ana"),
    (2, "Bruno"),
    (3, "Carla"),
    (4, "Diego"),
], ["id_aluno", "nome_aluno"])

# Renomeando 'nome' para 'nome_disciplina'
disciplinas = spark.createDataFrame([
    (101, "Probabilidade e Estatística", "Obrigatoria", 60),
    (102, "Banco de Dados", "Obrigatoria", 60),
    (103, "Projeto e Análise de Algoritos", "Obrigatoria", 60),
    (104, "Gestão de Tempo", "Optativa", 30),
],["id_disciplina", "nome_disciplina", "tipo", "limite_vagas"] )

inscricoes = spark.createDataFrame([
    (1, 101), (2, 101), (3, 101),
    (1, 102), (2, 102),
    (1, 103), (2, 103), (3, 103),
], ["id_aluno", "id_disciplina"])

ocupacao = (
    inscricoes.groupBy("id_disciplina")
    .agg(F.countDistinct("id_aluno").alias("inscritos"))
    .join(disciplinas, "id_disciplina", "left")
    .withColumn(
        "vagas_disponiveis",
        F.col("limite_vagas") - F.col("inscritos")
    )
)

print("OCUPACAO DAS DISCIPLINAS")
ocupacao.select(
    "nome_disciplina",
    "inscritos",
    "vagas_disponiveis"
).show()

detalhes = (
    inscricoes
    .join(alunos, "id_aluno")
    .join(disciplinas, "id_disciplina")
)

por_aluno = (
    detalhes.groupBy("id_aluno","nome_aluno")
    .agg(
        F.sum(
            F.when(F.col("tipo") == "Obrigatoria", 1)
             .otherwise(0)
        ).alias("obrigatorias"),
        F.sum(
            F.when(F.col("tipo") == "Optativa", 1)
             .otherwise(0)
        ).alias("optativas")
    )
).withColumn(
    "matricula_valida",
    (F.col("obrigatorias") <= 4) &
    (F.col("optativas") <= 2)
)

print("validacao por aluno")
por_aluno.show()

situacao = (
    ocupacao.withColumn(
        "situacao",
        F.when(F.col("inscritos") < 3, "CANCELADA")
         .otherwise("CONFIRMADA")
    )
)

print("situacao das disciplinas")
situacao.select(
        "nome_disciplina", "inscritos","situacao"
).show()

confirmadas = situacao.filter(
    F.col("situacao") == "CONFIRMADA"
)

cobrancas = (
    inscricoes
    .join(
        confirmadas.select("id_disciplina"),
        "id_disciplina"
    )
    .join(alunos, "id_aluno")
    .join(
        disciplinas.select("id_disciplina", "nome_disciplina"),
        "id_disciplina"
    )
)

OCUPACAO DAS DISCIPLINAS
+--------------------+---------+-----------------+
|     nome_disciplina|inscritos|vagas_disponiveis|
+--------------------+---------+-----------------+
|Projeto e Análise...|        3|               57|
|Probabilidade e E...|        3|               57|
|      Banco de Dados|        2|               58|
+--------------------+---------+-----------------+

validacao por aluno
+--------+----------+------------+---------+----------------+
|id_aluno|nome_aluno|obrigatorias|optativas|matricula_valida|
+--------+----------+------------+---------+----------------+
|       2|     Bruno|           3|        0|            true|
|       1|       Ana|           3|        0|            true|
|       3|     Carla|           2|        0|            true|
+--------+----------+------------+---------+----------------+

situacao das disciplinas
+--------------------+---------+----------+
|     nome_disciplina|inscritos|  situacao|
+--------------------+---------+----------+
|Proj

In [ ]:
#Analise do aluno
%pip install -q gradio

In [ ]:
import gradio as gr
import pandas as pd
from pyspark.sql import functions as F

In [ ]:
alunos_web = spark.createDataFrame([
    ("11111111111", "Ana Silva", "Engenharia de Computacao",
     "Sim", "Pendente", "PROT-001", "Calculo I"),
    ("22222222222", "Bruno Lima", "Administracao",
     "Nao", "Sim", "PROT-002", "Nenhuma"),
], [
    "cpf", "aluno", "curso", "impressao_contrato",
    "impressao_certificado", "protocolos",
    "disciplinas_canceladas"
])

In [ ]:
Protocolos = spark.createDataFrame([
    ("11111111111", "PROT-001", "Documentacao",
     "Em analise", "Documento pendente", "Pendente"),
    ("22222222222", "PROT-002", "Matricula",
     "Concluido", "Solicitacao aprovada", "Deferido"),
], [
    "cpf", "protocolo", "resultado",
    "situacao", "comentario", "parecer_final"
])

In [ ]:
livros = spark.createDataFrame([
    ("11111111111", "Introducao a Computacao", "2026-10-25"),
    ("11111111111", "Banco de Dados", "2026-10-30"),
    ("22222222222", "Gestao Empresarial", "2026-11-05"),
], ["cpf", "livro", "data_devolucao"])

In [ ]:
def consultar_aluno(cpf_digitado):
    # Limpa o CPF mantendo apenas números
    cpf = "".join(c for c in str(cpf_digitado) if c.isdigit())

    if len(cpf) != 11:
        vazio = pd.DataFrame()
        return (
            "Informe um CPF válido com 11 dígitos.",
            vazio, vazio, vazio
        )

    # Filtra as informações nas tabelas do Spark e converte para Pandas
    df_aluno = alunos_web.filter(F.col("cpf") == cpf).toPandas()

    if df_aluno.empty:
        vazio = pd.DataFrame()
        return (
            f"Aluno com CPF {cpf} não foi encontrado.",
            vazio, vazio, vazio
        )

    df_protocolos = Protocolos.filter(F.col("cpf") == cpf).toPandas()
    df_livros = livros.filter(F.col("cpf") == cpf).toPandas()

    # Retorna mensagem de sucesso e as tabelas correspondentes
    nome_aluno = df_aluno["aluno"].values[0]
    mensagem = f"Consulta realizada com sucesso para: {nome_aluno}"

    return mensagem, df_aluno, df_protocolos, df_livros

# Criando a interface gráfica no Gradio
with gr.Blocks() as app:
    gr.Markdown("#  Sistema de Consulta Acadêmica (Spark + Gradio)")
    gr.Markdown("Digite o CPF do aluno (apenas números) para consultar seus dados acadêmicos, protocolos e pendências de livros.")

    with gr.Row():
        cpf_input = gr.Textbox(label="CPF do Aluno", placeholder="Ex: 11111111111")
        btn_consultar = gr.Button(" Consultar", variant="primary")

    status_output = gr.Markdown(value="*Aguardando consulta...*")

    gr.Markdown("###  Informações do Aluno")
    aluno_output = gr.DataFrame()

    gr.Markdown("###  Protocolos Registrados")
    protocolos_output = gr.DataFrame()

    gr.Markdown("###  Livros Emprestados")
    livros_output = gr.DataFrame()

    btn_consultar.click(
        fn=consultar_aluno,
        inputs=[cpf_input],
        outputs=[status_output, aluno_output, protocolos_output, livros_output]
    )

# Inicializa a interface gráfica no notebook
app.launch(quiet=True, share=True)

* Running on public URL: https://d7c9b896e1c615007e.gradio.live


In [ ]:
#